## Implement firecrawl mcp server


In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import (
    MCPServerStdio,
    create_static_tool_filter,
    MCPServerStreamableHttp,
)
import os
from pathlib import Path
from datetime import datetime
from IPython.display import Markdown, display

load_dotenv(override=True)


### Version 1 - Streamable HTTP


In [ ]:
FIRECRAWL_API_KEY = os.environ["FIRECRAWL_API_KEY"]

firecrawl_params = {
    "url": f"https://mcp.firecrawl.dev/{FIRECRAWL_API_KEY}/v2/mcp",
}


async with MCPServerStreamableHttp(
    name="firecrawl",  # this is mainly to identify mcp servers in traces and logs
    params=firecrawl_params,
) as server:
    tools = await server.list_tools()

    for tool in tools:
        print(tool.name)


### Version 2 - STDio (standard input / output)


In [ ]:
firecrawl_params = {
    "command": "npx",
    "args": ["-y", "firecrawl-mcp"],
    "env": {
        "FIRECRAWL_API_KEY": FIRECRAWL_API_KEY,
    },
}

In [ ]:
async with MCPServerStdio(
    params=firecrawl_params, client_session_timeout_seconds=60
) as server:
    firecrawl_tools = await server.list_tools()

firecrawl_tools


In [ ]:
instructions = "You search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Nvidia stock price and briefly summarize its outlook. For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-5.4-mini"
search_only = create_static_tool_filter(allowed_tool_names=["firecrawl_search"])


In [ ]:
async with MCPServerStdio(
    params=firecrawl_params, client_session_timeout_seconds=60, tool_filter=search_only
) as mcp_server:
    agent = Agent(
        name="firecrawl_agent",
        instructions=instructions,
        model=model,
        mcp_servers=[mcp_server],
    )
    with trace("firecrawl"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))
